# Inflation différenciée en France: tableau comparatif 2026

Ce notebook reprend la logique du mémoire en appliquant les prix de l'Insee d'août 2026 aux paniers de consommation de Budget de famille 2017.

Les résultats sont des calculs à paniers fixes, et non des indices catégoriels publiés par l'Insee.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analyse import (
    CATEGORY_SPECS, DIVISION_LABELS, DIVISION_SHORT_LABELS, SHARE_COLUMNS,
    build_comparison_table, calculate_category, load_price_ratios,
)

divisions, ratios, metadata = load_price_ratios()
all_results, all_details = [], []
for category, spec in CATEGORY_SPECS.items():
    result, detail = calculate_category(category, spec, ratios)
    all_results.append(result)
    all_details.append(detail)
results = pd.concat(all_results, ignore_index=True)
details = pd.concat(all_details, ignore_index=True)
comparison = build_comparison_table(results, details)
division_rates = details.groupby('division')['division_rate'].first().to_dict()

## Toutes les catégories

**Définition des déciles:** ils partagent la distribution des niveaux de vie en dix groupes de même taille, du plus faible au plus élevé. Le décile 1 correspond aux 10 % situés en bas et le décile 10 aux 10 % situés en haut.

Pour chaque grand poste, l'en-tête rappelle son inflation nationale entre août 2025 et août 2026. La cellule indique sa part dans le budget du profil.

In [ ]:
tableau = comparison[[
    'dimension', 'group_label', 'modeled_inflation',
    'difference_vs_modeled_total', *SHARE_COLUMNS.values()
]].copy()

poste_headers = {
    SHARE_COLUMNS[code]: f"{DIVISION_SHORT_LABELS[code]} ({division_rates[code]:+.2f} %)"
    for code in DIVISION_LABELS
}
tableau = tableau.rename(columns={
    'dimension': 'Dimension',
    'group_label': 'Catégorie',
    'modeled_inflation': 'Inflation modélisée (%)',
    'difference_vs_modeled_total': 'Écart au panier moyen (point)',
    **poste_headers,
})
formats = {
    'Inflation modélisée (%)': '{:.2f}',
    'Écart au panier moyen (point)': '{:+.2f}',
    **{header: '{:.2f} %' for header in poste_headers.values()},
}
tableau.style.format(formats)

## Comparaison dimension par dimension

In [ ]:
for dimension, sous_tableau in tableau.groupby('Dimension', sort=False):
    print(f'\n{dimension}')
    display(sous_tableau.drop(columns='Dimension').reset_index(drop=True).style.format(formats))

## Repères

In [ ]:
pd.Series(metadata, name='Valeur').to_frame()

Les dimensions se recoupent: un même ménage peut être rural, locataire, ouvrier et appartenir à un décile de niveau de vie. Les lignes ne doivent donc pas être additionnées entre elles. Le panier moyen modélisé ne reproduit pas exactement l'IPC officiel, car il conserve les structures de dépenses de 2017.